# CrimeScope ML — 05. Score & Serve v3 (Lakehouse)

**Description:** Load 4 registered models (log1p, sqrt, violent, property), score tracts
using an ensemble (avg of log1p + sqrt), compute blended risk scores, SHAP drivers, and
enriched output table.

**v3 changes:**
- Loads sqrt model and ensembles with log1p model
- Uses pruned feature list for violent/property sub-models
- Reads model config from notebook 04's saved config

**Tables written:**
- `tract_risk_scores` — enriched per-tract risk output
- `tract_risk_scores_history` — append-only audit trail

**Prerequisites:** `04` (4 trained models in Unity Catalog, model_config_v3.json).

In [ ]:
%pip install lightgbm shap "mlflow==2.14.3" "opentelemetry-sdk==1.22.0" "opentelemetry-api==1.22.0" "opentelemetry-semantic-conventions==0.43b0" -q

In [ ]:
import json
from datetime import datetime

import mlflow
import numpy as np
import pandas as pd
import shap
from pyspark.sql import functions as F
from scipy.stats import percentileofscore

spark.sql("USE CATALOG varanasi")
spark.sql("USE SCHEMA default")

FEATURES_TABLE = "varanasi.default.tract_crime_features"
ACS_TABLE = "varanasi.default.tract_acs_population"
TRACTS_TABLE = "varanasi.default.cook_tract_boundaries"
SCORES_TABLE = "varanasi.default.tract_risk_scores"
HISTORY_TABLE = "varanasi.default.tract_risk_scores_history"
VOLUME_PATH = "/Volumes/varanasi/default/ml_data"

# Load config saved by notebook 04
with open(f"{VOLUME_PATH}/model_config_v3.json") as f:
    config = json.load(f)

FEATURE_COLS = config["feature_cols"]
FEATURE_COLS_PRUNED = config["feature_cols_pruned"]
BEST_STRATEGY = config["best_strategy"]

MODEL_NAME = config["model_names"]["overall_log"]
MODEL_NAME_SQRT = config["model_names"]["overall_sqrt"]
MODEL_NAME_V = config["model_names"]["violent"]
MODEL_NAME_P = config["model_names"]["property"]

FEATURE_LABELS = {
    "lag_1m_count": "Last month's crime count",
    "lag_2m_count": "Crime count 2 months ago",
    "lag_3m_count": "Crime count 3 months ago",
    "lag_6m_count": "Crime count 6 months ago",
    "lag_12m_count": "Crime count 12 months ago",
    "rolling_mean_3m": "3-month crime average",
    "rolling_mean_6m": "6-month crime average",
    "rolling_mean_12m": "12-month crime average",
    "rolling_std_6m": "6-month crime volatility",
    "rolling_max_6m": "6-month peak crime count",
    "rolling_min_6m": "6-month lowest crime count",
    "rolling_std_12m": "12-month crime volatility",
    "rolling_max_12m": "12-month peak crime count",
    "rolling_min_12m": "12-month lowest crime count",
    "mom_change": "Month-over-month change",
    "violent_lag_1m": "Last month's violent crimes",
    "violent_lag_3m": "Violent crimes 3 months ago",
    "violent_lag_6m": "Violent crimes 6 months ago",
    "violent_rolling_3m": "3-month violent crime trend",
    "violent_rolling_6m": "6-month violent crime trend",
    "violent_rolling_12m": "12-month violent crime average",
    "violent_ratio": "Violent crime proportion",
    "violent_ratio_6m": "6-month violent crime ratio",
    "property_lag_1m": "Last month's property crimes",
    "property_lag_3m": "Property crimes 3 months ago",
    "property_lag_6m": "Property crimes 6 months ago",
    "property_rolling_3m": "3-month property crime trend",
    "property_rolling_6m": "6-month property crime trend",
    "property_rolling_12m": "12-month property crime average",
    "month_of_year": "Month of year (seasonality)",
    "month_sin": "Seasonal cycle (sine)",
    "month_cos": "Seasonal cycle (cosine)",
    "year": "Year",
    "same_month_last_year": "Same month last year",
    "yoy_change": "Year-over-year change",
    "total_pop_acs": "Total population",
    "median_hh_income_acs": "Median household income",
    "poverty_rate_acs": "Neighborhood poverty rate",
    "housing_units_acs": "Housing unit count",
    "log_pop": "Population (log scale)",
    "log_income": "Income (log scale)",
    "crime_rate_per_1k": "Crime rate per 1,000 residents",
    "poverty_x_crime": "Poverty \u00d7 crime interaction",
    "income_crime_ratio": "Income-to-crime ratio",
    "pop_per_housing_unit": "Population density (per housing unit)",
    "city_month_total": "City-wide monthly crime total",
    "city_total_lag1": "City-wide crime (previous month)",
    "tract_share_of_city": "Tract's share of city crime",
    "tract_vs_city_avg": "Tract vs city average ratio",
    "ca_avg_crime": "Community area average crime",
    "tract_vs_ca_avg": "Tract vs community area average",
    "trend_accel": "Trend acceleration",
    "cv_6m": "6-month volatility coefficient",
    "cv_12m": "12-month volatility coefficient",
    "trend_3m": "3-month trend direction",
}

print(f"Models: {MODEL_NAME} + {MODEL_NAME_SQRT} (ensemble)")
print(f"Sub-models: {MODEL_NAME_V}, {MODEL_NAME_P}")
print(f"Features: {len(FEATURE_COLS)} (full) | {len(FEATURE_COLS_PRUNED)} (pruned for sub-models)")
print(f"Ensemble strategy selected by notebook 04: {BEST_STRATEGY}")

---
## 1. Load all 4 models from Unity Catalog

In [ ]:
mlflow.set_registry_uri("databricks-uc")

model_log = mlflow.lightgbm.load_model(f"models:/{MODEL_NAME}@champion")
model_sqrt = mlflow.lightgbm.load_model(f"models:/{MODEL_NAME_SQRT}@champion")
model_violent = mlflow.lightgbm.load_model(f"models:/{MODEL_NAME_V}@champion")
model_property = mlflow.lightgbm.load_model(f"models:/{MODEL_NAME_P}@champion")

print(f"Log1p model:    best_iter={model_log.best_iteration_}")
print(f"Sqrt model:     best_iter={model_sqrt.best_iteration_}")
print(f"Violent model:  best_iter={model_violent.best_iteration_}")
print(f"Property model: best_iter={model_property.best_iteration_}")

---
## 2. Load latest-month features

In [ ]:
feat_sdf = spark.table(FEATURES_TABLE)

max_month = feat_sdf.select(F.max("month_start")).first()[0]
print(f"Scoring for month: {max_month}")

latest_sdf = feat_sdf.filter(F.col("month_start") == max_month)
latest_sdf = latest_sdf.filter(F.col("lag_1m_count").isNotNull())

for col in FEATURE_COLS:
    latest_sdf = latest_sdf.withColumn(col, F.coalesce(F.col(col), F.lit(0.0)))

select_cols = ["tract_geoid", "month_start", "community_area"] + FEATURE_COLS + [
    "y_incidents_12m", "y_violent_12m", "y_property_12m",
    "violent_count", "property_count",
]
latest_pdf = latest_sdf.select(*select_cols).toPandas()
latest_pdf["incident_count"] = latest_pdf["lag_1m_count"]

print(f"Tracts to score: {len(latest_pdf)}")

---
## 3. Predict with ensemble (log1p + sqrt) + sub-models

In [ ]:
X = latest_pdf[FEATURE_COLS].astype(float).fillna(0)
X_pruned = latest_pdf[FEATURE_COLS_PRUNED].astype(float).fillna(0)

# Ensemble: average log1p and sqrt predictions
pred_log = np.maximum(np.expm1(np.maximum(model_log.predict(X), 0)), 0)
pred_sqrt = np.maximum(np.square(np.maximum(model_sqrt.predict(X), 0)), 0)

if BEST_STRATEGY == "ensemble":
    latest_pdf["predicted_next_30d"] = np.round(0.5 * pred_log + 0.5 * pred_sqrt, 2)
elif BEST_STRATEGY == "sqrt":
    latest_pdf["predicted_next_30d"] = np.round(pred_sqrt, 2)
else:
    latest_pdf["predicted_next_30d"] = np.round(pred_log, 2)

# Violent sub-model (pruned features)
pred_v_log = model_violent.predict(X_pruned)
latest_pdf["predicted_violent_30d"] = np.maximum(np.expm1(np.maximum(pred_v_log, 0)), 0).round(2)

# Property sub-model (pruned features)
pred_p_log = model_property.predict(X_pruned)
latest_pdf["predicted_property_30d"] = np.maximum(np.expm1(np.maximum(pred_p_log, 0)), 0).round(2)

# Weighted rules baseline
latest_pdf["baseline_predicted"] = np.maximum(
    0.30 * X["rolling_mean_3m"].fillna(0) +
    0.25 * X["rolling_mean_12m"].fillna(0) +
    0.20 * X["lag_1m_count"].fillna(0) +
    0.15 * X["same_month_last_year"].fillna(X["rolling_mean_12m"].fillna(0)) +
    0.10 * (X["city_month_total"].fillna(0) * X["tract_share_of_city"].fillna(0)),
    0
).round(2)

print(f"Predictions complete (strategy: {BEST_STRATEGY})")
print(f"  Overall  \u2014 mean: {latest_pdf['predicted_next_30d'].mean():.1f}, max: {latest_pdf['predicted_next_30d'].max():.1f}")
print(f"  Violent  \u2014 mean: {latest_pdf['predicted_violent_30d'].mean():.1f}, max: {latest_pdf['predicted_violent_30d'].max():.1f}")
print(f"  Property \u2014 mean: {latest_pdf['predicted_property_30d'].mean():.1f}, max: {latest_pdf['predicted_property_30d'].max():.1f}")

---
## 4. Blended risk scoring

In [ ]:
def blended_risk_score(predictions, pct_weight=0.70, abs_weight=0.30):
    preds = np.array(predictions)
    pct_scores = np.array([percentileofscore(preds, v, kind="rank") for v in preds])
    log_preds = np.log1p(preds)
    log_min, log_max = log_preds.min(), log_preds.max()
    if log_max > log_min:
        abs_scores = (log_preds - log_min) / (log_max - log_min) * 100
    else:
        abs_scores = np.full_like(log_preds, 50.0)
    return np.round(pct_weight * pct_scores + abs_weight * abs_scores, 1)

latest_pdf["risk_score"] = blended_risk_score(latest_pdf["predicted_next_30d"].values)
latest_pdf["violent_score"] = blended_risk_score(latest_pdf["predicted_violent_30d"].values)
latest_pdf["property_score"] = blended_risk_score(latest_pdf["predicted_property_30d"].values)

latest_pdf["risk_tier"] = pd.cut(
    latest_pdf["risk_score"],
    bins=[0, 25, 50, 75, 90, 100],
    labels=["Low", "Moderate", "Elevated", "High", "Critical"],
    include_lowest=True,
)

latest_pdf["model_vs_baseline"] = (
    (latest_pdf["predicted_next_30d"] - latest_pdf["baseline_predicted"]) /
    latest_pdf["baseline_predicted"].clip(lower=0.1)
).round(3)

latest_pdf["trend_direction"] = np.where(
    latest_pdf["predicted_next_30d"] > latest_pdf["lag_1m_count"] * 1.05, "rising",
    np.where(latest_pdf["predicted_next_30d"] < latest_pdf["lag_1m_count"] * 0.95, "falling", "stable")
)

print(f"Scored {len(latest_pdf)} tracts")
print(f"\nRisk tier distribution:")
print(latest_pdf["risk_tier"].value_counts().sort_index())
print(f"\nTrend distribution:")
print(latest_pdf["trend_direction"].value_counts())

---
## 5. SHAP top-5 drivers per tract

In [ ]:
explainer = shap.TreeExplainer(model_log)
shap_values = explainer.shap_values(X)

def top_drivers(shap_row, feature_names, feature_vals, n=5):
    abs_vals = np.abs(shap_row)
    top_idx = abs_vals.argsort()[-n:][::-1]
    drivers = []
    for idx in top_idx:
        feat = feature_names[idx]
        drivers.append({
            "feature": feat,
            "label": FEATURE_LABELS.get(feat, feat.replace("_", " ").title()),
            "shap_value": round(float(shap_row[idx]), 4),
            "feature_value": round(float(feature_vals[idx]), 4),
            "direction": "up" if shap_row[idx] > 0 else "down",
        })
    return json.dumps(drivers)

latest_pdf["top_drivers_json"] = [
    top_drivers(shap_values[i], FEATURE_COLS, X.iloc[i].values) for i in range(len(shap_values))
]

top1 = pd.Series([FEATURE_COLS[np.abs(shap_values[i]).argmax()] for i in range(len(shap_values))])
print(f"SHAP top-1 driver diversity ({top1.nunique()} unique features):")
print(top1.value_counts().head(10))

print(f"\nExample (highest risk tract):")
top_row = latest_pdf.sort_values("risk_score", ascending=False).iloc[0]
print(f"  Tract: {top_row['tract_geoid']}  Score: {top_row['risk_score']}")
print(f"  Drivers: {top_row['top_drivers_json']}")

---
## 6. Build final output table

In [ ]:
acs_pdf = spark.table(ACS_TABLE).select(
    "tract_geoid", "total_pop_acs", "median_hh_income_acs",
    "poverty_rate_acs", "housing_units_acs"
).toPandas()

tracts_pdf = spark.table(TRACTS_TABLE).select(
    "tract_geoid", "NAMELSAD", "wkt"
).toPandas()

output_cols = [
    "tract_geoid", "month_start",
    "predicted_next_30d", "predicted_violent_30d", "predicted_property_30d",
    "baseline_predicted",
    "risk_score", "violent_score", "property_score", "risk_tier",
    "model_vs_baseline", "trend_direction",
    "incident_count", "y_incidents_12m", "y_violent_12m", "y_property_12m",
    "lag_1m_count", "rolling_mean_3m", "rolling_mean_12m",
    "violent_ratio", "violent_ratio_6m",
    "top_drivers_json",
]

scores = latest_pdf[output_cols].copy()
scores = scores.merge(acs_pdf, on="tract_geoid", how="left")
scores = scores.merge(tracts_pdf[["tract_geoid", "NAMELSAD"]], on="tract_geoid", how="left")
scores["scored_at"] = datetime.utcnow().isoformat() + "Z"
scores["model_name"] = MODEL_NAME
scores["model_version"] = "v3"

print(f"Output table: {len(scores)} rows, {len(scores.columns)} columns")
display(scores.sort_values("risk_score", ascending=False).head(10))

---
## 7. Write `tract_risk_scores` Delta table

In [ ]:
scores_sdf = spark.createDataFrame(scores)

spark.sql(f"DROP TABLE IF EXISTS {SCORES_TABLE}")
scores_sdf.write.saveAsTable(SCORES_TABLE)
spark.sql(f"OPTIMIZE {SCORES_TABLE} ZORDER BY (tract_geoid)")
spark.sql(f"""COMMENT ON TABLE {SCORES_TABLE} IS
  'v3 enriched per-tract risk scores with ensemble + sub-scores + SHAP drivers'""")

n = spark.table(SCORES_TABLE).count()
print(f"\n\u2705 Wrote {n} tract scores to {SCORES_TABLE} (Delta, Z-ordered)")

---
## 8. Append to history table

In [ ]:
try:
    spark.table(HISTORY_TABLE)
except Exception:
    scores_sdf.limit(0).write.saveAsTable(HISTORY_TABLE)

scores_sdf.write.mode("append").option("mergeSchema", "true").saveAsTable(HISTORY_TABLE)

total_history = spark.table(HISTORY_TABLE).count()
n_runs = spark.table(HISTORY_TABLE).select("scored_at").distinct().count()
print(f"History table: {total_history:,} total rows across {n_runs} scoring runs")

---
## 9. Verification + summary

In [ ]:
print("=" * 60)
print("CrimeScope ML Pipeline v3 \u2014 Complete")
print("=" * 60)

tables = [
    ("varanasi.default.chicago_crimes_raw", "Raw crime incidents"),
    ("varanasi.default.cook_tract_boundaries", "Census tract polygons"),
    ("varanasi.default.chicago_crimes_with_tract", "Crimes \u2192 tract spatial join"),
    ("varanasi.default.chicago_crime_monthly_by_tract", "Monthly tract counts"),
    ("varanasi.default.tract_acs_population", "ACS population/income"),
    ("varanasi.default.tract_crime_features", "Feature table (50+ features)"),
    (SCORES_TABLE, "v3 ensemble risk scores"),
    (HISTORY_TABLE, "Historical scores audit trail"),
]

print("\nDelta Tables in Unity Catalog:")
for table, desc in tables:
    try:
        n = spark.table(table).count()
        print(f"  \u2713 {table:55s} {n:>10,} rows  \u2014 {desc}")
    except Exception:
        print(f"  \u2717 {table:55s} {'MISSING':>10}  \u2014 {desc}")

print(f"\nMLflow Models (v3):")
print(f"  Overall (log1p): {MODEL_NAME}")
print(f"  Overall (sqrt):  {MODEL_NAME_SQRT}")
print(f"  Violent:         {MODEL_NAME_V}")
print(f"  Property:        {MODEL_NAME_P}")
print(f"  Ensemble:        {BEST_STRATEGY}")

scores_check = spark.table(SCORES_TABLE)
display(scores_check.select(
    F.count("*").alias("n_tracts"),
    F.round(F.avg("risk_score"), 1).alias("avg_risk"),
    F.round(F.avg("violent_score"), 1).alias("avg_violent"),
    F.round(F.avg("property_score"), 1).alias("avg_property"),
    F.round(F.avg("predicted_next_30d"), 1).alias("avg_pred_30d"),
    F.min("risk_score").alias("min_score"),
    F.max("risk_score").alias("max_score"),
))

print("\n\u2705 Pipeline complete. Run notebook 06 to export for backend.")